# Anthology poem length comparison

Compares average poem text length across anthologies. An anthology with a much higher average likely has boundary detection issues (poems being merged); a much lower average suggests truncation or missed extractions.

In [ ]:
FILES = [
    {
        "name": f"englishpoets{n:02d}wardiala",
        "json": f"pipeline_output/marker_poems/englishpoets{n:02d}wardiala_poems.json",
    }
    for n in range(1, 6)
]

# Anthologies whose mean char count deviates from the overall mean by more
# than this fraction are flagged as outliers (e.g. 0.5 = 50%)
OUTLIER_THRESHOLD = 0.5

In [ ]:
import json
from pathlib import Path

import pandas as pd


def load_json(name, path):
    with open(path, encoding="utf-8") as f:
        poems = json.load(f)
    rows = []
    for p in poems:
        text = p.get("text", "")
        rows.append({
            "anthology": name,
            "title": p.get("title", ""),
            "source_page": p.get("source_page", 0),
            "chars": len(text),
            "words": len(text.split()),
            "lines": len([l for l in text.splitlines() if l.strip()]),
        })
    return pd.DataFrame(rows)


dfs = []
for f in FILES:
    dfs.append(load_json(f["name"], f["json"]))
    print(f"{f['name']}: {len(dfs[-1])} poems loaded")

df = pd.concat(dfs, ignore_index=True)

## Per-anthology summary

In [ ]:
summary = (
    df.groupby("anthology")
    .agg(
        poems=("chars", "count"),
        mean_chars=("chars", "mean"),
        median_chars=("chars", "median"),
        mean_words=("words", "mean"),
        mean_lines=("lines", "mean"),
    )
    .reset_index()
)

overall_mean = df["chars"].mean()
summary["deviation"] = (summary["mean_chars"] - overall_mean) / overall_mean
summary["flagged"] = summary["deviation"].abs() > OUTLIER_THRESHOLD

display(
    summary.style
    .format({
        "mean_chars":   "{:.0f}",
        "median_chars": "{:.0f}",
        "mean_words":   "{:.0f}",
        "mean_lines":   "{:.0f}",
        "deviation":    "{:+.1%}",
    })
    .apply(lambda row: ["background-color: #ffe0e0" if row["flagged"] else "" for _ in row], axis=1)
)

print(f"\nOverall mean chars per poem: {overall_mean:.0f}")

## Distribution of poem lengths per anthology

In [ ]:
percentiles = (
    df.groupby("anthology")["chars"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .drop(columns="count")
    .reset_index()
)

display(percentiles.style.format({c: "{:.0f}" for c in percentiles.columns if c != "anthology"}))

## Flagged anthologies

In [ ]:
flagged = summary[summary["flagged"]][["anthology", "poems", "mean_chars", "deviation"]]

if flagged.empty:
    print(f"No anthologies exceed the {OUTLIER_THRESHOLD:.0%} deviation threshold.")
else:
    print(f"{len(flagged)} anthology/anthologies flagged (threshold: ±{OUTLIER_THRESHOLD:.0%}):")
    display(flagged.style.format({"mean_chars": "{:.0f}", "deviation": "{:+.1%}"}))